# Avance 02 — INEGI ATUS — Etapa 3: reglas y conservación
**Equipo A&C | Inteligencia de Negocios**

Trabajamos con **62,424 registros** delimitados de 2018 a 2025 para las 16 alcaldías de CDMX. La Etapa 2 identificó **145 duplicados exactos adicionales** entre las 46 columnas originales y **3 fechas inválidas**. No eliminamos registros automáticamente: conservamos los originales, agregamos columnas derivadas y exportamos evidencias para auditoría.

**Unidad de observación:** registro de ATUS. La igualdad de atributos no prueba por sí sola que dos registros sean un único hecho de tránsito.
**Guía de lectura:** cada bloque explica *qué hacemos*, *por qué lo hacemos* y *qué evidencia obtenemos*. Las reglas y resultados esperados se mantienen iguales a la versión que ya ejecutamos.

## 1. Localización de la entrada y contrato de lectura

**Objetivo:** localizar automáticamente el CSV delimitado y comprobar que trabajamos con la misma versión de los datos.

**Decisión de calidad:** mantenemos los 46 campos originales y distinguimos las columnas auxiliares creadas en la Etapa 1. Si falta una columna necesaria, detenemos el proceso en lugar de continuar con datos incompletos.

**Salida esperada:** rutas de entrada y salida, 46 columnas originales y 62,424 filas esperadas.

In [1]:
# PASO 1. Importamos herramientas para manejar rutas, contar datos y usar DataFrames.
# Path evita escribir rutas absolutas que cambian entre computadoras.
# Counter se conserva como parte del entorno del notebook original.
# pandas permite leer, comparar y exportar tablas; numpy está disponible para operaciones numéricas.
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
from IPython.display import display

# Buscamos el archivo de la Etapa 1 desde la carpeta actual y sus carpetas padre.
# Esto funciona aunque abramos JupyterLab dentro de notebooks/INEGI_ATUS.
ETAPA1 = None
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidata = base / 'resultados' / 'resultados_INEGI_ATUS' / 'etapa1'
    if (candidata / 'INEGI_ATUS_etapa1.csv').is_file():
        ETAPA1 = candidata
        break
if ETAPA1 is None:
    raise FileNotFoundError('No encontramos INEGI_ATUS_etapa1.csv en resultados/resultados_INEGI_ATUS/etapa1')
# El archivo de entrada se lee, pero nunca se sobrescribe.
# La salida de esta etapa se guarda en una carpeta independiente.
ENTRADA = ETAPA1 / 'INEGI_ATUS_etapa1.csv'
SALIDA = ETAPA1.parent / 'etapa3'
SALIDA.mkdir(parents=True, exist_ok=True)
# Leemos la cantidad de filas aptas declarada en el resumen de la Etapa 1.
ESPERADAS = int(pd.read_csv(ETAPA1 / 'resumen_delimitacion_INEGI_ATUS.csv').loc[0, 'filas_aptas'])
COLUMNAS = pd.read_csv(ENTRADA, nrows=0, encoding='utf-8-sig').columns.tolist()
# Distinguimos las 46 columnas originales de las columnas auxiliares
# que fueron añadidas durante la delimitación.
ORIGINALES = [c for c in COLUMNAS if c not in ['archivo_origen','fila_origen','alcaldia_normalizada','motivo_exclusion']]
# Categorías de víctimas que utilizaremos para construir indicadores derivados.
# Las listas separan fallecidos y lesionados; no modificamos sus columnas fuente.
MUERTOS = ['CONDMUERTO','PASAMUERTO','PEATMUERTO','CICLMUERTO','OTROMUERTO','NEMUERTO']
HERIDOS = ['CONDHERIDO','PASAHERIDO','PEATHERIDO','CICLHERIDO','OTROHERIDO','NEHERIDO']
REQUERIDAS = ['ANIO','MES','ID_DIA','alcaldia_normalizada'] + MUERTOS + HERIDOS
# Detenemos la ejecución si falta una columna imprescindible.
# Es preferible avisar del cambio de esquema antes que generar resultados incorrectos.
faltantes = [c for c in REQUERIDAS if c not in COLUMNAS]
if faltantes:
    raise KeyError('Faltan columnas: ' + ', '.join(faltantes))
# Contrato de lectura: esperamos el mismo esquema y número de filas ya validado.
assert len(ORIGINALES) == 46, f'Esperábamos 46 columnas originales; encontramos {len(ORIGINALES)}'
assert ESPERADAS == 62424, f'La Etapa 1 reporta {ESPERADAS} filas; revisar versión del proyecto.'
print('Entrada:', ENTRADA)
print('Salida:', SALIDA)
print('Columnas originales:', len(ORIGINALES))
print('Registros esperados:', ESPERADAS)

Entrada: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa1/INEGI_ATUS_etapa1.csv
Salida: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa3
Columnas originales: 46
Registros esperados: 62424


## 2. Reglas de transformación

1. **Duplicados:** calculamos huellas de las 46 columnas originales y marcamos apariciones adicionales; **no retiramos filas**. Exportamos todas las filas que pertenecen a grupos idénticos.
2. **Fechas:** creamos `fecha_validada` con conversión estricta a calendario; los valores imposibles se dejan vacíos únicamente en la columna derivada.
3. **Personas:** validamos cada columna de víctimas como entero no negativo y calculamos `total_fallecidos_validado` y `total_lesionados_validado`. Si alguna categoría es inválida, el total derivado se deja sin valor, evitando sumas parciales.
4. **Alcaldías:** conservamos `alcaldia_normalizada` de Etapa 1; no modificamos claves originales.
5. **Trazabilidad:** mantenemos todas las columnas de entrada y añadimos indicadores, sin sobrescribir archivos previos.

### Justificación de nuestras decisiones

- **No deduplicamos por apariencia:** dos registros con las mismas 46 variables pueden requerir revisión de la fuente antes de afirmar que corresponden al mismo accidente.
- **Separamos los tipos de duplicados:** «duplicados adicionales» cuenta las apariciones después de la primera; «filas en grupos idénticos» cuenta todas las filas implicadas.
- **No imputamos fechas:** si una combinación de año, mes y día no forma una fecha real, registramos una alerta y dejamos vacía solo la fecha derivada.
- **No corregimos víctimas sin evidencia:** si los campos originales fueran inválidos, evitamos calcular totales parciales que puedan parecer completos.

Estas decisiones son **provisionales y auditables**; no sustituyen una validación con la metodología y los identificadores oficiales de ATUS.

## 3. Procesamiento y evidencia de grupos repetidos

**Qué haremos:** crearemos una copia del DataFrame; calcularemos grupos idénticos usando las 46 columnas originales; construiremos fechas válidas y totales de víctimas derivados.

**Qué no haremos:** no borraremos filas, no reemplazaremos las fechas originales ni cambiaremos las cifras originales de personas lesionadas o fallecidas.

**Evidencia esperada:** 145 duplicados adicionales, 3 fechas inválidas y 62,424 registros conservados.

In [2]:
# PASO 3. Leemos la tabla delimitada como texto para no alterar ceros iniciales
# ni convertir automáticamente claves, folios o códigos en números.
# keep_default_na=False mantiene las celdas vacías como cadenas vacías.
# Hacemos una copia profunda: todos los cambios posteriores se aplican a
# datos_trabajo, nunca al DataFrame de entrada ni al CSV de la Etapa 1.
# Son 62,424 registros: podemos analizarlos en memoria sin alterar el archivo de entrada.
datos_entrada = pd.read_csv(ENTRADA, dtype=str, encoding='utf-8-sig', keep_default_na=False)
datos_trabajo = datos_entrada.copy(deep=True)

# Huellas sobre las 46 columnas originales; verificamos también igualdad exacta
# dentro de cada grupo de huella para descartar colisiones de hash.
# Una huella (hash) resume los valores de una fila y facilita encontrar coincidencias.
# Solo consideramos las 46 columnas originales, no las columnas auxiliares.
huellas = pd.util.hash_pandas_object(datos_trabajo[ORIGINALES], index=False)
datos_trabajo['huella_fila_original'] = huellas.astype(str)
conteos_huellas = huellas.value_counts()
# mascara_grupo incluye TODAS las filas que pertenecen a grupos repetidos.
# mascara_adicional marca solo las apariciones posteriores a la primera.
# Por eso ambos conteos no significan lo mismo.
mascara_grupo = huellas.map(conteos_huellas).gt(1)
mascara_adicional = datos_trabajo.duplicated(subset=ORIGINALES, keep='first')
datos_trabajo['grupo_identico_original'] = mascara_grupo.to_numpy()
datos_trabajo['duplicado_adicional_original'] = mascara_adicional.to_numpy()
# La coincidencia de huella debe corresponder a la comparación exacta de las 46 columnas.
# Confirmamos los grupos usando la comparación exacta de pandas:
# si las huellas coincidieran por accidente, esta prueba lo detectaría.
mascara_grupo_exacta = datos_trabajo.duplicated(subset=ORIGINALES, keep=False)
assert (mascara_grupo.to_numpy() == mascara_grupo_exacta.to_numpy()).all(), 'Revisar posible colisión de huellas.'

# Fecha derivada: usamos valores originales sin sustituirlos.
# Construimos una fecha de texto a partir de año, mes y día.
# errors="coerce" convierte fechas imposibles en NaT (no en una fecha inventada).
# Guardamos la versión validada en una NUEVA columna.
fecha_texto = (datos_trabajo['ANIO'].str.strip() + '-' +
               datos_trabajo['MES'].str.strip().str.zfill(2) + '-' +
               datos_trabajo['ID_DIA'].str.strip().str.zfill(2))
fecha = pd.to_datetime(fecha_texto, format='%Y-%m-%d', errors='coerce')
datos_trabajo['fecha_validada'] = fecha.dt.strftime('%Y-%m-%d').fillna('')
datos_trabajo['fecha_calendario_invalida'] = fecha.isna()

# Revisamos cada categoría de víctimas: debe ser numérica, entera y >= 0.
# Creamos indicadores booleanos por columna para documentar su validez.
for c in MUERTOS + HERIDOS:
    num = pd.to_numeric(datos_trabajo[c].str.strip(), errors='coerce')
    valida = num.notna() & num.ge(0) & num.mod(1).eq(0)
    datos_trabajo[c + '_valido'] = valida

# Sumamos las categorías por registro, por separado para fallecidos y lesionados.
# Si alguna categoría no es válida, dejamos el total derivado como dato faltante.
# Int64 (con I mayúscula) permite enteros y valores faltantes en pandas.
for columnas, destino in [(MUERTOS, 'total_fallecidos_validado'),
                           (HERIDOS, 'total_lesionados_validado')]:
    nums = datos_trabajo[columnas].apply(pd.to_numeric, errors='coerce')
    validas = datos_trabajo[[c + '_valido' for c in columnas]].all(axis=1)
    totales = nums.sum(axis=1).where(validas)
    datos_trabajo[destino] = totales.astype('Int64')

print('Filas:', len(datos_trabajo))
print('Duplicados adicionales exactos:', int(mascara_adicional.sum()))
print('Filas en grupos idénticos:', int(mascara_grupo.sum()))
print('Fechas inválidas:', int(fecha.isna().sum()))
# Comparamos con los hallazgos de la Etapa 2: 145 duplicados adicionales
# y 3 fechas inválidas. Una diferencia obliga a revisar la fuente de entrada.
assert int(mascara_adicional.sum()) == 145, 'El conteo de duplicados difiere de la Etapa 2; revisar la versión de entrada.'
assert int(fecha.isna().sum()) == 3, 'El conteo de fechas inválidas difiere de la Etapa 2; revisar la versión de entrada.' 

Filas: 62424
Duplicados adicionales exactos: 145
Filas en grupos idénticos: 253
Fechas inválidas: 3


## 4. Exportación de datos conservados y muestras de revisión

Exportamos tres productos distintos para no confundir la tabla final con las alertas de revisión.

| Archivo | Propósito |
|---|---|
| `INEGI_ATUS_etapa3_conservados.csv` | Conserva todos los registros con indicadores nuevos |
| `INEGI_ATUS_grupos_identicos_revision.csv` | Facilita revisar todas las filas pertenecientes a grupos idénticos |
| `INEGI_ATUS_fechas_invalidas_revision.csv` | Separa los registros cuyas fechas derivadas no pudieron validarse |

También guardamos `impacto_reglas_INEGI_ATUS.csv` para registrar los conteos y las columnas añadidas. **Una fila en un CSV de revisión no es una fila eliminada.**

In [3]:
# PASO 4. Definimos tres archivos diferentes:
# 1) tabla procesada completa; 2) filas de grupos idénticos;
# 3) casos con fechas que no pudieron validarse.
# Los CSV de evidencia son subconjuntos para revisión, no registros eliminados.
DESTINO = SALIDA / 'INEGI_ATUS_etapa3_conservados.csv'
REPETIDOS = SALIDA / 'INEGI_ATUS_grupos_identicos_revision.csv'
FECHAS = SALIDA / 'INEGI_ATUS_fechas_invalidas_revision.csv'

# Exportamos siempre a una ruta nueva, sin alterar los CSV previos.
datos_trabajo.to_csv(DESTINO, index=False, encoding='utf-8-sig')
datos_trabajo.loc[mascara_grupo].to_csv(REPETIDOS, index=False, encoding='utf-8-sig')
datos_trabajo.loc[fecha.isna()].to_csv(FECHAS, index=False, encoding='utf-8-sig')

# Resumimos el efecto de las reglas: filas, alertas y columnas agregadas.
# Este cuadro sirve como evidencia para el Avance 02 y la bitácora.
impacto = pd.DataFrame([{
    'filas_recibidas': len(datos_entrada),
    'filas_exportadas': len(datos_trabajo),
    'filas_eliminadas': 0,
    'duplicados_adicionales_46_columnas': int(mascara_adicional.sum()),
    'filas_en_grupos_identicos': int(mascara_grupo.sum()),
    'grupos_identicos': int((conteos_huellas > 1).sum()),
    'fechas_calendario_invalidas': int(fecha.isna().sum()),
    'totales_fallecidos_sin_validar': int(datos_trabajo['total_fallecidos_validado'].isna().sum()),
    'totales_lesionados_sin_validar': int(datos_trabajo['total_lesionados_validado'].isna().sum()),
    'columnas_entrada': len(COLUMNAS),
    'columnas_salida': len(datos_trabajo.columns),
}])
impacto.to_csv(SALIDA / 'impacto_reglas_INEGI_ATUS.csv', index=False, encoding='utf-8-sig')
display(impacto)
print('CSV principal:', DESTINO)
print('Grupos repetidos:', REPETIDOS)
print('Fechas para revisión:', FECHAS)

,filas_recibidas,filas_exportadas,filas_eliminadas,duplicados_adicionales_46_columnas,filas_en_grupos_identicos,grupos_identicos,fechas_calendario_invalidas,totales_fallecidos_sin_validar,totales_lesionados_sin_validar,columnas_entrada,columnas_salida
0,62424,62424,0,145,253,108,3,0,0,50,69


CSV principal: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa3/INEGI_ATUS_etapa3_conservados.csv
Grupos repetidos: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa3/INEGI_ATUS_grupos_identicos_revision.csv
Fechas para revisión: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa3/INEGI_ATUS_fechas_invalidas_revision.csv


## 5. Comprobaciones finales

**Objetivo:** demostrar que el procesamiento no perdió filas, mantuvo intactas las columnas de entrada y exportó las evidencias.

La comprobación de columnas compara la copia procesada con el DataFrame leído originalmente. El conteo de salida vuelve a abrir el CSV para verificar que se escribió completo. Si una prueba no se cumple, `assert` detiene el notebook para que investiguemos antes de entregar los resultados.

In [4]:
# PASO 5. Reabrimos la salida por bloques para comprobar su número real de filas.
# Así no dependemos solamente del conteo del DataFrame que está en memoria.
# Cada comprobación devuelve True/False y permite auditar una regla específica.
# Verificamos el archivo exportado y los conteos sin modificar el CSV original.
conteo_salida = sum(len(b) for b in pd.read_csv(DESTINO, chunksize=20000, dtype=str))
# Verificamos conteos, conservación de columnas, disponibilidad de archivos
# y exportación de evidencias antes de declarar terminada la etapa.
comprobaciones_df = pd.DataFrame([
    {'comprobacion':'mismas_filas_que_etapa1', 'cumple':len(datos_entrada) == ESPERADAS},
    {'comprobacion':'sin_filas_eliminadas', 'cumple':len(datos_trabajo) == len(datos_entrada)},
    {'comprobacion':'salida_tiene_todas_las_filas', 'cumple':conteo_salida == ESPERADAS},
    {'comprobacion':'duplicados_adicionales_esperados', 'cumple':int(mascara_adicional.sum()) == 145},
    {'comprobacion':'fechas_invalidas_esperadas', 'cumple':int(fecha.isna().sum()) == 3},
    {'comprobacion':'columnas_originales_conservadas', 'cumple':datos_trabajo[COLUMNAS].equals(datos_entrada[COLUMNAS])},
    {'comprobacion':'entrada_sigue_disponible', 'cumple':ENTRADA.is_file()},
    {'comprobacion':'evidencias_exportadas', 'cumple':all(p.is_file() for p in [DESTINO, REPETIDOS, FECHAS])},
])
comprobaciones_df.to_csv(SALIDA / 'comprobaciones_INEGI_ATUS_etapa3.csv', index=False, encoding='utf-8-sig')
display(comprobaciones_df)
assert comprobaciones_df['cumple'].all(), 'Revisar las comprobaciones no satisfechas.'
print('Etapa 3 INEGI–ATUS: comprobaciones superadas; no se eliminaron filas.')

,comprobacion,cumple
0,mismas_filas_que_etapa1,True
1,sin_filas_eliminadas,True
2,salida_tiene_todas_las_filas,True
3,duplicados_adicionales_esperados,True
4,fechas_invalidas_esperadas,True
5,columnas_originales_conservadas,True
6,entrada_sigue_disponible,True
7,evidencias_exportadas,True


Etapa 3 INEGI–ATUS: comprobaciones superadas; no se eliminaron filas.


## 6. Registro para nuestra bitácora

Durante la Etapa 3 revisamos las alertas identificadas en INEGI–ATUS: **145 apariciones adicionales en grupos de filas idénticas** y **3 fechas inválidas**. Conservamos los 62,424 registros porque no contamos con evidencia suficiente para asegurar que las filas idénticas representen eventos duplicados. Agregamos indicadores de calidad y fechas validadas, calculamos totales derivados de lesionados y fallecidos, y exportamos los casos que requieren revisión. Mantuvimos intacto el archivo de la Etapa 1 y documentamos las comprobaciones y el impacto de nuestras reglas.

**Pendiente:** revisar los grupos idénticos y la cobertura de cada año antes de interpretar tendencias o deduplicar registros.

### Interpretación para nuestra entrega

- **Frecuencia:** podemos usar la tabla conservada para contar registros por alcaldía y periodo, sin afirmar que el número de registros sea igual al riesgo vial.
- **Severidad:** los totales derivados de lesionados y fallecidos permiten preparar indicadores descriptivos, sujetos a las definiciones de ATUS.
- **Duplicados:** documentamos 145 apariciones adicionales idénticas, pero la decisión de retirarlas queda pendiente de revisión.
- **Trazabilidad:** el archivo de la Etapa 1 permanece disponible y las salidas de la Etapa 3 se guardan aparte.

**Criterio de cierre:** consideramos ejecutada la Etapa 3 cuando las ocho comprobaciones muestran `True`; esto valida la consistencia del procedimiento, no resuelve por sí solo todas las dudas metodológicas.